In [1]:
import sys
import asyncio

# Fix for Windows issues in Jupyter notebooks
if sys.platform == "win32":
    # 1. Use ProactorEventLoop for subprocess support
    if not isinstance(asyncio.get_event_loop_policy(), asyncio.WindowsProactorEventLoopPolicy):
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())

    # 2. Redirect stderr to avoid fileno() error when launching MCP servers
    if "ipykernel" in sys.modules:
        sys.stderr = sys.__stderr__

# 1.加载环境变量
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage
from langchain_mcp_adapters.client import MultiServerMCPClient

load_dotenv()

# 2.初始化模型
model = init_chat_model(
    model="deepseek-v4-flash",
    extra_body={"thinking": {"type": "disabled"}}
)

# 3.定义工具，用MCP获取工具

# 3.1.定义mcp client
client = MultiServerMCPClient(
    {
        "time-mcp": {
            "transport": "stdio",
            "args": [
                "-y",
                "time-mcp"
            ],
            "command": "npx"
        },
        # 直接复制魔搭社区代码, 这里用type直接报错,url过期的话代码也会直接红x
        # "12306-mcp": {
        #     "type": "streamable_http",
        #     "url": "https://mcp.api-inference.modelscope.net/20dcf42db7d347/mcp"
        # },
        "12306-mcp": {
          "transport": "streamable_http",
          "url": "https://mcp.api-inference.modelscope.net/20dcf42db7d347/mcp"
        }
    }
)
# 3.2.用client拉取tool
tools = await client.get_tools()

# 4.创建Agent，绑定模型和工具
agent = create_agent(
    model=model,
    tools=tools
)

# 5.由于MCP的Tool是异步的，所以必须用ainvoke调用Agent，是异步调用
response = await agent.ainvoke(
    {"messages": [HumanMessage("帮我查一下下周四从北京到杭州的高铁")]}
)
print(response)

{'messages': [HumanMessage(content='帮我查一下下周四从北京到杭州的高铁', additional_kwargs={}, response_metadata={}, id='04f8337f-73f1-412b-ba08-becc60f613e0'), AIMessage(content='我来帮您查询下周四从北京到杭州的高铁车票。首先，让我获取当前日期来确定下周四的具体日期。', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 93, 'prompt_tokens': 3191, 'total_tokens': 3284, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 896, 'image_tokens': None, 'text_tokens': None}, 'prompt_cache_hit_tokens': 896, 'prompt_cache_miss_tokens': 2295}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'a26a7955944dc5c60445bff77fac9c8e', 'id': '8c7fa943-d61d-45ad-865d-97634a60b5f5', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a047c9-578b-7661-bf80-7c8d51af2812-0', tool_calls=[{'name': 'get-current-date', 'args': {}, 'id': 'call_00_EPXc6nRbFaetnuGKhyLd8359', 'type': 'tool_call'}, {'name': 'get-s